# Error angle and distance to bead data from smooth data

In [8]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Save error angle and distance data csv

In [9]:
#Input dir- folder path of smooth csv, output dir- where the chase metric csv should be saved
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [13]:
# Resultant error
def compute_error_angle_head(bead, head, center):
    v_heading = head - center
    v_bead    = bead - head
    dot = []
    cross = []
    for b, h in zip(v_heading, v_bead):
        d = np.dot(b, h)
        c = np.cross(b, h)
        dot.append(d)
        cross.append(c)
    dot   = np.array(dot)
    cross = np.linalg.norm(cross, axis=1)

    angle           = np.degrees (np.arctan2(cross,dot))
    return angle

# Horizontal error (XY plane)
def compute_hori_error(bead, head, center):
    v_heading    = head - center
    v_bead       = bead - head
    v_heading_xy = v_heading[:, 0:2]
    v_bead_xy    = v_bead[:, 0:2]
    dot   = np.sum(v_heading_xy * v_bead_xy, axis=1)
    cross = v_heading_xy[:, 0] * v_bead_xy[:, 1] - v_heading_xy[:, 1] * v_bead_xy[:, 0]
  
    angle           = np.degrees (np.arctan2(np.abs(cross),dot))
    return angle

# Vertical error (YZ plane)
def compute_ver_error(bead, head, center):
    v_heading    = head - center
    v_bead       = bead - head
    v_heading_yz = v_heading[:, 1:3]
    v_bead_yz    = v_bead[:, 1:3]
    dot   = np.sum(v_heading_yz * v_bead_yz, axis=1)
    cross = v_heading_yz[:, 0] * v_bead_yz[:, 1] - v_heading_yz[:, 1] * v_bead_yz[:, 0]
    
    angle           = np.degrees (np.arctan2(np.abs(cross),dot))
    return angle

In [14]:
# Find smooth files
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv"))
if not csv_files:
    raise FileNotFoundError("No SMOOTH CSV files found.")

for path in csv_files:
    fname = os.path.basename(path)
    print("Processing:", fname)
    df = pd.read_csv(path)
    # Use frame directly from CSV
    frames = df["frame"].values
    bead   = df[["pt1_X","pt1_Y","pt1_Z"]].values
    head   = df[["pt2_X","pt2_Y","pt2_Z"]].values
    center = df[["center_X","center_Y","center_Z"]].values
    
    # Distances
    dist_head = np.linalg.norm(bead - head, axis=1)
    
    # Error angles
    err_head  = compute_error_angle_head(bead, head, center)
    err_hori  = compute_hori_error(bead, head, center)
    err_vert  = compute_ver_error(bead, head, center)
    
    # Save CSV
    out_df = pd.DataFrame({
        "frame":                  frames,
        "dist_head_m":            dist_head,
        "error_angle_head_deg":   err_head,
        "error_angle_horiz_deg":  err_hori,
        "error_angle_vert_deg":   err_vert,
    })
    
    out_name = fname.replace("_SMOOTH.csv", "_CHASE_METRICS.csv")
    out_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)
    print("Saved:", out_name)

print("\nChase metrics CSVs saved.")

Processing: Trial2_180rpmxyzpts_SMOOTH.csv
Saved: Trial2_180rpmxyzpts_CHASE_METRICS.csv
Processing: Trial2_200rpmxyzpts_SMOOTH.csv
Saved: Trial2_200rpmxyzpts_CHASE_METRICS.csv
Processing: Trial3_180rpmxyzpts_SMOOTH.csv
Saved: Trial3_180rpmxyzpts_CHASE_METRICS.csv
Processing: Trial4_400rpmxyzpts_SMOOTH.csv
Saved: Trial4_400rpmxyzpts_CHASE_METRICS.csv
Processing: Trial5_180rpmxyzpts_SMOOTH.csv
Saved: Trial5_180rpmxyzpts_CHASE_METRICS.csv
Processing: Trial5_400rpmxyzpts_SMOOTH.csv
Saved: Trial5_400rpmxyzpts_CHASE_METRICS.csv
Processing: Trial7_400rpmxyzpts_SMOOTH.csv
Saved: Trial7_400rpmxyzpts_CHASE_METRICS.csv

Chase metrics CSVs saved.
